In [15]:
import glob
import os
import re
import json
from pathlib import Path
import pandas as pd
import numpy as np
from docx import Document

# Cell 0 - Flexible data extraction optimized for Generative AI
# Handles variable document structures by extracting full text + flexible section mapping

file_pattern = "*_SIPReport.docx"
paths = sorted(glob.glob(file_pattern))

if not paths:
    print("No DOCX files found. Checking current directory:")
    print(os.getcwd())
    print("Files in directory:", os.listdir())
    raise SystemExit("No SIP reports found matching pattern: " + file_pattern)

print(f"Found {len(paths)} SIP report files")

def extract_docx_flexible(docx_path):
    """
    Flexible extraction that captures:
    1. Full text (for LLM processing)
    2. All section headings found
    3. Section-to-content mapping (what sections exist in this specific document)
    4. Basic metadata (title, company, date hints)
    
    Key: Does NOT force content into fixed categories.
    Instead, documents what sections actually exist.
    """
    try:
        doc = Document(docx_path)
        
        # Extract all text with style information
        paragraphs_with_style = []
        for para in doc.paragraphs:
            text = para.text.strip()
            if text:
                paragraphs_with_style.append({
                    'text': text,
                    'style': para.style.name
                })
        
        # Build full text from all paragraphs
        full_text = " ".join([p['text'] for p in paragraphs_with_style])
        
        # Extract section structure (Heading 1 and Heading 2 as hierarchy)
        sections_found = []
        current_heading_1 = None
        
        for p in paragraphs_with_style:
            if p['style'] == "Heading 1":
                current_heading_1 = p['text']
            elif p['style'] == "Heading 2":
                sections_found.append({
                    'level': 2,
                    'title': p['text'],
                    'parent': current_heading_1
                })
            elif p['style'] == "Heading 3":
                sections_found.append({
                    'level': 3,
                    'title': p['text'],
                    'parent': current_heading_1
                })
        
        # Extract basic metadata from content
        metadata = {
            'total_paragraphs': len(paragraphs_with_style),
            'total_words': len(full_text.split()),
            'has_tables': len(doc.tables) > 0,
            'num_tables': len(doc.tables),
        }
        
        # Try to find company name (often appears early in document)
        company_match = re.search(r'(?:company|organisation|organization|firm)[:\s]+([^\n.]+)', full_text, re.IGNORECASE)
        if company_match:
            metadata['company'] = company_match.group(1).strip()[:100]
        else:
            metadata['company'] = "Not specified"
        
        # Try to find student name/ID patterns
        for p in paragraphs_with_style[:20]:  # Check first 20 paragraphs
            if re.search(r'\bTP\d{6}|\bStudent ID|\bMatric', p['text'], re.IGNORECASE):
                metadata['has_student_id'] = True
                break
        
        return {
            'full_text': full_text,
            'sections': sections_found,
            'metadata': metadata,
            'total_content_length': len(full_text)
        }
    except Exception as e:
        print(f"Error reading {docx_path}: {e}")
        return {
            'full_text': "",
            'sections': [],
            'metadata': {'error': str(e), 'company': "Error"},
            'total_content_length': 0
        }

# Load and process all DOCX files
records = []
section_inventory = {}  # Track all sections found across all documents

for docx_path in paths:
    try:
        filename = os.path.basename(docx_path)
        extraction = extract_docx_flexible(docx_path)
        
        # Create record for generative AI - store full text for AI processing
        record = {
            "_source_file": filename,
            "_file_path": docx_path,
            "full_text": extraction['full_text'],  # Keep full text (will save separately)
            "total_words": extraction['metadata'].get('total_words', 0),
            "total_paragraphs": extraction['metadata'].get('total_paragraphs', 0),
            "has_tables": extraction['metadata'].get('has_tables', False),
            "company_mentioned": extraction['metadata'].get('company', "Not specified"),
            "sections_json": json.dumps(extraction['sections']),
            "section_count": len(extraction['sections']),
        }
        
        records.append(record)
        
        # Inventory all sections found
        for section in extraction['sections']:
            section_title = section['title']
            if section_title not in section_inventory:
                section_inventory[section_title] = []
            section_inventory[section_title].append(filename)
        
        print(f"✓ {filename:25s} | {extraction['metadata'].get('total_words', 0):5d} words | {len(extraction['sections']):2d} sections | {extraction['metadata'].get('company', 'N/A')[:20]}")
        
    except Exception as e:
        print(f"✗ Failed to process {filename}: {e}")

if not records:
    raise SystemExit("No records extracted from any DOCX files.")

# Create main DataFrame - CSV version without full_text (it's too long for CSV)
df_csv = []
for r in records:
    r_copy = r.copy()
    r_copy['full_text'] = r_copy['full_text'][:500]  # Truncate to 500 chars for CSV preview
    df_csv.append(r_copy)

df = pd.DataFrame(df_csv)

# Save CSV version (preview only)
csv_out_path = "SIP_reports_cleaned.csv"
df.to_csv(csv_out_path, index=False)
print(f"\n✓ CSV preview saved to: {csv_out_path}")

# Create full dataset with complete text as JSON (better for AI)
df_full = pd.DataFrame(records)

json_out_path = "SIP_reports_full.json"
full_data = []
for idx, row in df_full.iterrows():
    full_data.append({
        'report_id': idx,
        'source_file': row['_source_file'],
        'full_text': row['full_text'],
        'metadata': {
            'total_words': int(row['total_words']),
            'total_paragraphs': int(row['total_paragraphs']),
            'has_tables': bool(row['has_tables']),
            'company': row['company_mentioned'],
            'section_count': int(row['section_count']),
        },
        'sections': json.loads(row['sections_json'])
    })

with open(json_out_path, 'w', encoding='utf-8') as f:
    json.dump(full_data, f, indent=2, ensure_ascii=False)

print(f"✓ Full dataset saved to: {json_out_path}")

# Save section inventory
section_inventory_path = "section_inventory.json"
with open(section_inventory_path, "w") as f:
    json.dump(section_inventory, f, indent=2)

print(f"✓ Section inventory saved to: {section_inventory_path}")

print("\n--- Section Inventory Across All Reports ---")
print(f"Total unique sections found: {len(section_inventory)}\n")

section_inventory_sorted = sorted(section_inventory.items(), key=lambda x: len(x[1]), reverse=True)
for section_title, filenames in section_inventory_sorted:
    coverage = (len(filenames) / len(df)) * 100
    preview = section_title[:60] + "..." if len(section_title) > 60 else section_title
    print(f"  [{coverage:5.1f}%] {preview:65s} | {len(filenames):2d} reports")

print(f"\n=== SUMMARY ===")
print(f"  - Total reports: {len(df)}")
print(f"  - CSV (preview): {csv_out_path}")
print(f"  - Full JSON (AI): {json_out_path}")
print(f"  - Ready for: LLM APIs, sentiment analysis, skill extraction")

Found 10 SIP report files
✓ 10_SIPReport.docx         |  4840 words | 55 sections | that there is NO pla
✓ 1_SIPReport.docx          |  1622 words |  0 sections | Name:  Gadgets Galax
✓ 2_SIPReport.docx          |  2617 words |  0 sections | Not specified
✓ 3_SIPReport.docx          |  4821 words |  0 sections | you worked for the b
✓ 4_SIPReport.docx          |  2652 words |  0 sections | Chilli Api Catering,
✓ 5_SIPReport.docx          |  2507 words |  0 sections | Changi General Hospi
✓ 6_SIPReport.docx          |  4674 words | 31 sections | Structure EssilorLux
✓ 7_SIPReport.docx          |  7264 words | 12 sections | that there is NO pla
✓ 8_SIPReport.docx          |  6003 words |  0 sections | that there is NO pla
✓ 9_SIPReport.docx          |  1614 words |  0 sections | The company is DB Sc

✓ CSV preview saved to: SIP_reports_cleaned.csv
✓ Full dataset saved to: SIP_reports_full.json
✓ Section inventory saved to: section_inventory.json

--- Section Inventory Across All Reports 

In [16]:
# Cell 1 - Validate cleaned data and understand document structure

print("=== VALIDATION: Flexible Document Structure ===\n")

# Reload the cleaned data
df_clean = pd.read_csv("SIP_reports_cleaned.csv")

print("Dataset Overview:")
print(f"  Shape: {df_clean.shape}")
print(f"  Reports: {len(df_clean)}")
print(f"  Columns: {len(df_clean.columns)}\n")

print("Column Structure:")
for i, col in enumerate(df_clean.columns, 1):
    dtype = df_clean[col].dtype
    non_null = df_clean[col].notna().sum()
    print(f"  {i:2d}. {col:25s} | Type: {str(dtype):10s} | Non-null: {non_null}/{len(df_clean)}")

print("\n" + "="*70)
print("Data Quality Check:")
print("="*70)
print(f"  Total records: {len(df_clean)}")
print(f"  Duplicate rows: {df_clean.duplicated().sum()}")
print(f"  Missing full_text: {df_clean['full_text'].isna().sum()}")

# Statistics on content
print(f"\nContent Statistics:")
print(f"  Avg words per report: {df_clean['total_words'].mean():.0f}")
print(f"  Avg sections per report: {df_clean['section_count'].mean():.1f}")
print(f"  Reports with tables: {df_clean['has_tables'].sum()} / {len(df_clean)}")

# Load section inventory
try:
    with open("section_inventory.json", "r") as f:
        section_inv = json.load(f)
    
    print(f"\nSection Coverage:")
    print(f"  Unique sections found: {len(section_inv)}")
    print(f"  Most common sections:")
    
    section_coverage = [(s, len(r)) for s, r in section_inv.items()]
    section_coverage.sort(key=lambda x: x[1], reverse=True)
    
    for section, count in section_coverage[:10]:
        pct = (count / len(df_clean)) * 100
        preview = section[:55] + "..." if len(section) > 55 else section
        print(f"    - {preview:60s}: {count} reports ({pct:5.1f}%)")
except:
    print("  (Section inventory file not yet generated)")

print("\n" + "="*70)
print("Sample Data:")
print("="*70)

for idx in range(min(2, len(df_clean))):
    row = df_clean.iloc[idx]
    print(f"\nReport {idx+1}: {row['_source_file']}")
    print(f"  Company: {row['company_mentioned']}")
    print(f"  Words: {row['total_words']} | Sections: {row['section_count']} | Tables: {row['has_tables']}")
    
    # Show section structure
    try:
        sections = json.loads(row['sections_json'])
        if sections:
            print(f"  Sections found:")
            for s in sections[:5]:
                print(f"    - {s['title']}")
            if len(sections) > 5:
                print(f"    ... and {len(sections)-5} more")
    except:
        pass
    
    # Show text preview
    print(f"  Text preview: {row['full_text'][:150]}...")

print(f"\n✓ Dataset ready for generative AI processing!")

=== VALIDATION: Flexible Document Structure ===

Dataset Overview:
  Shape: (10, 9)
  Reports: 10
  Columns: 9

Column Structure:
   1. _source_file              | Type: str        | Non-null: 10/10
   2. _file_path                | Type: str        | Non-null: 10/10
   3. full_text                 | Type: str        | Non-null: 10/10
   4. total_words               | Type: int64      | Non-null: 10/10
   5. total_paragraphs          | Type: int64      | Non-null: 10/10
   6. has_tables                | Type: bool       | Non-null: 10/10
   7. company_mentioned         | Type: str        | Non-null: 10/10
   8. sections_json             | Type: str        | Non-null: 10/10
   9. section_count             | Type: int64      | Non-null: 10/10

Data Quality Check:
  Total records: 10
  Duplicate rows: 0
  Missing full_text: 0

Content Statistics:
  Avg words per report: 3861
  Avg sections per report: 9.8
  Reports with tables: 2 / 10

Section Coverage:
  Unique sections found: 86
  Most 

In [17]:
# Cell 2 - Prepare data for Generative AI: Create prompt templates and analysis stubs

print("=== GENERATIVE AI PREPARATION ===\n")

# Create a JSON export with full text for LLM API calls
ai_dataset = []
for idx, row in df_clean.iterrows():
    sections = json.loads(row['sections_json']) if pd.notna(row['sections_json']) else []
    section_titles = [s['title'] for s in sections]
    
    ai_record = {
        'id': idx,
        'source_file': row['_source_file'],
        'metadata': {
            'total_words': int(row['total_words']),
            'total_paragraphs': int(row['total_paragraphs']),
            'company': row['company_mentioned'],
            'has_tables': bool(row['has_tables']),
            'section_count': int(row['section_count']),
            'section_titles': section_titles
        },
        'full_text': row['full_text']
    }
    ai_dataset.append(ai_record)

# Save as JSON for LLM processing
ai_json_path = "SIP_reports_for_AI.json"
with open(ai_json_path, "w", encoding='utf-8') as f:
    json.dump(ai_dataset, f, indent=2, ensure_ascii=False)

print(f"✓ AI-optimized JSON saved to: {ai_json_path}")
print(f"  - Format: {len(ai_dataset)} records with full text + metadata")
print(f"  - Ready for: OpenAI API, Claude, local LLMs\n")

# Create example prompt templates for common use cases
prompt_templates = {
    "sentiment_analysis": """Analyze the sentiment of this SIP report. 
Report: {full_text}

Provide:
1. Overall sentiment (Positive/Neutral/Negative)
2. Key emotional indicators
3. Challenges mentioned
4. Growth areas highlighted""",
    
    "skill_extraction": """Extract all technical and professional skills mentioned in this report.
Report: {full_text}

Provide:
1. Technical skills (programming languages, tools, platforms)
2. Professional skills (communication, teamwork, leadership)
3. Soft skills developed
4. Skills gaps identified""",
    
    "company_analysis": """Analyze the company/organization described in this report.
Report: {full_text}

Provide:
1. Company industry and size
2. Key technologies used
3. Company culture observations
4. Organizational structure insights""",
    
    "learning_outcomes": """Summarize the learning outcomes from this internship report.
Report: {full_text}

Provide:
1. Major learnings
2. Personal growth areas
3. Applied knowledge
4. Future career implications"""
}

# Save prompt templates
with open("AI_prompt_templates.json", "w") as f:
    json.dump(prompt_templates, f, indent=2)

print(f"✓ Prompt templates saved to: AI_prompt_templates.json")
print(f"  Available templates:")
for template_name in prompt_templates.keys():
    print(f"    - {template_name}")

# Print structure for reference
print(f"\n=== How to use the AI datasets ===\n")
print(f"Option 1: OpenAI API")
print(f"  import json")
print(f"  with open('SIP_reports_for_AI.json') as f:")
print(f"      data = json.load(f)")
print(f"  for record in data:")
print(f"      # Use record['full_text'] with GPT prompts\n")

print(f"Option 2: Local Processing")
print(f"  df_ai = pd.read_csv('SIP_reports_cleaned.csv')")
print(f"  for idx, row in df_ai.iterrows():")
print(f"      text = row['full_text']")
print(f"      # Process with sentiment analysis, NER, etc.\n")

print(f"Option 3: Batch Analysis")
print(f"  Load SIP_reports_for_AI.json → send to LLM API")
print(f"  Parse responses → aggregate results by skill/company/sentiment")

print(f"\n✓ Datasets ready for AI analysis and LLM processing!")

=== GENERATIVE AI PREPARATION ===

✓ AI-optimized JSON saved to: SIP_reports_for_AI.json
  - Format: 10 records with full text + metadata
  - Ready for: OpenAI API, Claude, local LLMs

✓ Prompt templates saved to: AI_prompt_templates.json
  Available templates:
    - sentiment_analysis
    - skill_extraction
    - company_analysis
    - learning_outcomes

=== How to use the AI datasets ===

Option 1: OpenAI API
  import json
  with open('SIP_reports_for_AI.json') as f:
      data = json.load(f)
  for record in data:
      # Use record['full_text'] with GPT prompts

Option 2: Local Processing
  df_ai = pd.read_csv('SIP_reports_cleaned.csv')
  for idx, row in df_ai.iterrows():
      text = row['full_text']
      # Process with sentiment analysis, NER, etc.

Option 3: Batch Analysis
  Load SIP_reports_for_AI.json → send to LLM API
  Parse responses → aggregate results by skill/company/sentiment

✓ Datasets ready for AI analysis and LLM processing!


In [18]:
# Cell 3 - AI Usage Examples: How to apply LLM/AI to the cleaned data

print("=== GENERATIVE AI USAGE EXAMPLES ===\n")

# Load the prepared data
with open("SIP_reports_for_AI.json", "r", encoding='utf-8') as f:
    ai_data = json.load(f)

# Example 1: Skill extraction simulation (without actual API)
print("EXAMPLE 1: Skill Extraction")
print("-" * 70)
sample_record = ai_data[0]
print(f"Processing: {sample_record['source_file']}")
print(f"Metadata: {sample_record['metadata']['total_words']} words, {sample_record['metadata']['section_count']} sections")

# Simulate skill extraction by finding tech keywords
tech_keywords = {
    'programming': ['python', 'java', 'sql', 'javascript', 'c++', 'r', 'scala'],
    'data_tools': ['tableau', 'power bi', 'excel', 'spark', 'hadoop', 'pandas', 'numpy'],
    'cloud': ['aws', 'azure', 'gcp', 'docker', 'kubernetes'],
    'big_data': ['hadoop', 'spark', 'hive', 'pig', 'kafka']
}

text_lower = sample_record['full_text'].lower()
found_skills = {}

for category, keywords in tech_keywords.items():
    found = [kw for kw in keywords if kw in text_lower]
    if found:
        found_skills[category] = found

print(f"\nSkills detected (keyword-based):")
for category, skills in found_skills.items():
    print(f"  {category}: {', '.join(skills)}")

# Example 2: Section analysis
print(f"\n\nEXAMPLE 2: Document Structure Analysis")
print("-" * 70)
print(f"Sections found in {sample_record['source_file']}:")
for section_title in sample_record['metadata']['section_titles'][:10]:
    print(f"  • {section_title}")
if len(sample_record['metadata']['section_titles']) > 10:
    print(f"  ... and {len(sample_record['metadata']['section_titles']) - 10} more")

# Example 3: Cross-document analysis
print(f"\n\nEXAMPLE 3: Cross-Document Insights")
print("-" * 70)

companies = [r['metadata']['company'] for r in ai_data]
companies_clean = [c for c in companies if c != 'N/A']

print(f"Companies mentioned: {len(set(companies_clean))} unique")
print(f"  Top companies:")
from collections import Counter
company_counts = Counter(companies_clean)
for company, count in company_counts.most_common(5):
    print(f"    - {company}: {count} reports")

print(f"\nReport statistics:")
print(f"  Average length: {np.mean([r['metadata']['total_words'] for r in ai_data]):.0f} words")
print(f"  Average sections: {np.mean([r['metadata']['section_count'] for r in ai_data]):.1f}")
print(f"  Reports with tables: {sum([r['metadata']['has_tables'] for r in ai_data])} / {len(ai_data)}")

# Example 4: Batch processing template
print(f"\n\nEXAMPLE 4: Batch Processing Template (for API calls)")
print("-" * 70)
print(f"""
# Template for batch processing with LLM API:

import json

with open('SIP_reports_for_AI.json') as f:
    records = json.load(f)

results = []

for record in records:
    # Create prompt for LLM
    prompt = f\"\"\"Analyze this internship report and extract:
1. Student's main learning areas
2. Technical skills developed  
3. Company industry and size
4. Career implications

Report text:
{{record['full_text']}}

Format response as JSON.
\"\"\"
    
    # Call your LLM API (e.g., OpenAI, Claude, local model)
    # response = your_llm_api_call(prompt)
    
    results.append({{
        'source': record['source_file'],
        'analysis': response  # Would contain structured data from LLM
    }})

# Save batch results
with open('AI_analysis_results.json', 'w') as f:
    json.dump(results, f, indent=2)
""")

print(f"\n✓ Dataset optimized for flexible, AI-driven analysis!")
print(f"  Full text preserved for LLM processing")
print(f"  Variable document structure handled gracefully")
print(f"  Section metadata available for context-aware prompting")

=== GENERATIVE AI USAGE EXAMPLES ===

EXAMPLE 1: Skill Extraction
----------------------------------------------------------------------
Processing: 10_SIPReport.docx
Metadata: 4840 words, 55 sections

Skills detected (keyword-based):
  programming: r


EXAMPLE 2: Document Structure Analysis
----------------------------------------------------------------------
Sections found in 10_SIPReport.docx:
  • Introduction
  • Explain the objectives of the report.
  • Describe the information on the context of the report, such as brief details about the SIP Organisation and your internship role.
  • State the parameters on which the report is based.
  • Work Environment
  • Describe the location and premises of the SIP Organisation
  • Comment on the conduciveness of the environment for work
  • Observations on aspects of workplace safety, facilities and equipment, etc.
  • Organisation structure
  • Provide a brief history of the SIP Organisation
  ... and 45 more


EXAMPLE 3: Cross-Document I

In [19]:
# Cell 5 - Section-aware chunking for RAG indexing
# Splits by section headings first, then applies character chunking inside each section.

import json
import re
from dataclasses import dataclass
from pathlib import Path

try:
    from langchain.text_splitter import RecursiveCharacterTextSplitter
except ImportError:
    try:
        from langchain_text_splitters import RecursiveCharacterTextSplitter
    except ImportError:
        class RecursiveCharacterTextSplitter:
            def __init__(self, chunk_size=1000, chunk_overlap=100, separators=None):
                self.chunk_size = chunk_size
                self.chunk_overlap = chunk_overlap
                self.separators = separators or ["\n\n", "\n", ". ", " "]

            def _split_text(self, text):
                text = str(text or "")
                if not text.strip():
                    return []

                chunks = []
                start = 0
                step = max(1, self.chunk_size - self.chunk_overlap)

                while start < len(text):
                    end = min(len(text), start + self.chunk_size)
                    if end < len(text):
                        cut = max(
                            text.rfind("\n\n", start, end),
                            text.rfind("\n", start, end),
                            text.rfind(". ", start, end),
                            text.rfind(" ", start, end),
                        )
                        if cut > start + 100:
                            end = cut + 1
                    chunk = text[start:end].strip()
                    if chunk:
                        chunks.append(chunk)
                    if end >= len(text):
                        break
                    start = max(end - self.chunk_overlap, start + step)
                return chunks

            def split_documents(self, documents):
                split_docs = []
                for document in documents:
                    text = document.page_content if hasattr(document, "page_content") else document.get("page_content", "")
                    metadata = document.metadata if hasattr(document, "metadata") else document.get("metadata", {})
                    for part in self._split_text(text):
                        split_docs.append(SimpleDocument(page_content=part, metadata=dict(metadata)))
                return split_docs

@dataclass
class SimpleDocument:
    page_content: str
    metadata: dict

print("=== SECTION-AWARE RAG CHUNKING ===\n")

rag_dir = Path("rag_assets")
rag_dir.mkdir(exist_ok=True)

full_json_path = rag_dir / "SIP_reports_full.json"
if not full_json_path.exists():
    raise FileNotFoundError(f"Missing input file: {full_json_path}")

with full_json_path.open("r", encoding="utf-8") as f:
    records = json.load(f)

print(f"Loaded {len(records)} reports from {full_json_path.name}")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " "]
)

def build_section_blocks(full_text, sections):
    """Return ordered section blocks using heading titles as boundaries."""
    if not sections:
        return [{
            "section_index": 0,
            "section_title": "Full Document",
            "parent_section": None,
            "section_level": 0,
            "block_text": full_text,
        }]

    blocks = []
    search_start = 0
    ordered_sections = [s for s in sections if str(s.get("title", "")).strip()]

    for idx, section in enumerate(ordered_sections):
        title = str(section.get("title", "")).strip()
        parent = section.get("parent")
        level = section.get("level", 0)

        pattern = re.compile(re.escape(title), re.IGNORECASE)
        match = pattern.search(full_text, search_start)
        if not match:
            continue

        content_start = match.end()
        next_start = len(full_text)

        for next_section in ordered_sections[idx + 1:]:
            next_title = str(next_section.get("title", "")).strip()
            if not next_title:
                continue
            next_match = re.compile(re.escape(next_title), re.IGNORECASE).search(full_text, content_start)
            if next_match:
                next_start = next_match.start()
                break

        body_text = full_text[content_start:next_start].strip()
        block_text = f"{title}\n\n{body_text}" if body_text else title
        blocks.append({
            "section_index": idx,
            "section_title": title,
            "parent_section": parent,
            "section_level": level,
            "block_text": block_text,
        })

        search_start = content_start

    if not blocks:
        blocks.append({
            "section_index": 0,
            "section_title": "Full Document",
            "parent_section": None,
            "section_level": 0,
            "block_text": full_text,
        })

    return blocks

# Convert reports into section blocks first, then chunk inside each block.
section_documents = []
section_block_count = 0
for record in records:
    full_text = record.get("full_text", "")
    sections = record.get("sections", [])
    section_blocks = build_section_blocks(full_text, sections)
    section_block_count += len(section_blocks)

    for block in section_blocks:
        section_documents.append(SimpleDocument(
            page_content=block["block_text"],
            metadata={
                "source_file": record.get("source_file", ""),
                "report_id": record.get("report_id"),
                "company": record.get("metadata", {}).get("company", "Not specified"),
                "total_words": record.get("metadata", {}).get("total_words", 0),
                "section_count": record.get("metadata", {}).get("section_count", 0),
                "has_tables": record.get("metadata", {}).get("has_tables", False),
                "section_index": block["section_index"],
                "section_title": block["section_title"],
                "parent_section": block["parent_section"],
                "section_level": block["section_level"],
                "sections": sections,
            }
        ))

print(f"Created {section_block_count} section blocks from {len(records)} reports")

chunks = text_splitter.split_documents(section_documents)
print(f"Created {len(chunks)} chunks total")

chunk_records = []
for chunk_id, chunk in enumerate(chunks):
    chunk_records.append({
        "chunk_id": chunk_id,
        "source_file": chunk.metadata.get("source_file", ""),
        "report_id": chunk.metadata.get("report_id"),
        "company": chunk.metadata.get("company", "Not specified"),
        "section_title": chunk.metadata.get("section_title", "Full Document"),
        "parent_section": chunk.metadata.get("parent_section"),
        "section_index": chunk.metadata.get("section_index", 0),
        "section_level": chunk.metadata.get("section_level", 0),
        "section_count": chunk.metadata.get("section_count", 0),
        "has_tables": chunk.metadata.get("has_tables", False),
        "chunk_length": len(chunk.page_content),
        "chunk_text": chunk.page_content,
    })

chunk_json_path = rag_dir / "SIP_reports_section_chunks.json"
with chunk_json_path.open("w", encoding="utf-8") as f:
    json.dump(chunk_records, f, indent=2, ensure_ascii=False)

chunk_csv_path = rag_dir / "SIP_reports_section_chunks.csv"
pd.DataFrame(chunk_records).to_csv(chunk_csv_path, index=False, encoding="utf-8")

print(f"✓ Saved chunk JSON to: {chunk_json_path}")
print(f"✓ Saved chunk CSV to: {chunk_csv_path}")
print("\nChunking summary:")
print(f"  Reports: {len(records)}")
print(f"  Section blocks: {section_block_count}")
print(f"  Chunks: {len(chunk_records)}")
print("  Chunk size: 1000 characters")
print("  Chunk overlap: 100 characters")

if chunk_records:
    sample = chunk_records[0]
    print("\nSample chunk:")
    print(f"  Source: {sample['source_file']}")
    print(f"  Section: {sample['section_title']}")
    print(f"  Length: {sample['chunk_length']}")
    print(f"  Preview: {sample['chunk_text'][:200]}...")

print("\n✓ Section-aware chunked data is ready for vector indexing and RAG retrieval.")

=== SECTION-AWARE RAG CHUNKING ===

Loaded 10 reports from SIP_reports_full.json
Created 105 section blocks from 10 reports
Created 319 chunks total
✓ Saved chunk JSON to: rag_assets\SIP_reports_section_chunks.json
✓ Saved chunk CSV to: rag_assets\SIP_reports_section_chunks.csv

Chunking summary:
  Reports: 10
  Section blocks: 105
  Chunks: 319
  Chunk size: 1000 characters
  Chunk overlap: 100 characters

Sample chunk:
  Source: 10_SIPReport.docx
  Section: Introduction
  Length: 12
  Preview: Introduction...

✓ Section-aware chunked data is ready for vector indexing and RAG retrieval.


In [20]:
# Cell 6 - Build embeddings and a lightweight vector store for RAG
# Uses a pre-trained SentenceTransformer model for dense embeddings.

import truststore
truststore.inject_into_ssl()

import os
os.environ["HF_HUB_DISABLE_SSL_VERIFICATION"] = "1"
os.environ["CURL_CA_BUNDLE"] = ""
os.environ["REQUESTS_CA_BUNDLE"] = ""

import json
from pathlib import Path
import joblib
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors

rag_dir = Path("rag_assets")
chunk_json_path = rag_dir / "SIP_reports_section_chunks.json"
if not chunk_json_path.exists():
    raise FileNotFoundError(f"Missing chunk file: {chunk_json_path}")

with chunk_json_path.open("r", encoding="utf-8") as f:
    chunk_records = json.load(f)

if not chunk_records:
    raise ValueError("No chunk records available for embedding.")

print("=== EMBEDDINGS AND VECTOR STORE ===\n")
print(f"Loaded {len(chunk_records)} chunks from {chunk_json_path.name}")

chunk_texts = [record["chunk_text"] for record in chunk_records]
chunk_metadata = [{
    "chunk_id": record["chunk_id"],
    "source_file": record["source_file"],
    "report_id": record["report_id"],
    "company": record["company"],
    "section_title": record["section_title"],
    "parent_section": record["parent_section"],
    "section_index": record["section_index"],
    "section_level": record["section_level"],
    "section_count": record["section_count"],
    "has_tables": record["has_tables"],
    "chunk_length": record["chunk_length"],
} for record in chunk_records]

embedding_model_name = "all-MiniLM-L6-v2"
cache_folder = rag_dir / "hf_cache"
cache_folder.mkdir(exist_ok=True)
model = SentenceTransformer(embedding_model_name, cache_folder=str(cache_folder))
print(f"Loaded embedding model: {embedding_model_name}")

embedding_matrix = model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)
print(f"Built dense embedding matrix with shape: {embedding_matrix.shape}")

# Fit a cosine nearest-neighbor index for retrieval.
vector_store = NearestNeighbors(metric="cosine", algorithm="brute")
vector_store.fit(embedding_matrix)
print("Fitted cosine nearest-neighbor vector store")

# Persist artifacts inside rag_assets/.
model_info_path = rag_dir / "rag_embedding_model.json"
embeddings_path = rag_dir / "rag_embeddings.npy"
vector_store_path = rag_dir / "rag_vector_store.joblib"
metadata_path = rag_dir / "rag_chunk_metadata.json"

np.save(embeddings_path, embedding_matrix)
joblib.dump(vector_store, vector_store_path)
with metadata_path.open("w", encoding="utf-8") as f:
    json.dump(chunk_metadata, f, indent=2, ensure_ascii=False)
with model_info_path.open("w", encoding="utf-8") as f:
    json.dump({
        "embedding_model": embedding_model_name,
        "normalize_embeddings": True,
        "metric": "cosine",
        "chunk_count": len(chunk_records),
        "cache_folder": str(cache_folder)
    }, f, indent=2)

print(f"✓ Saved embedding model info to: {model_info_path}")
print(f"✓ Saved embeddings to: {embeddings_path}")
print(f"✓ Saved vector store to: {vector_store_path}")
print(f"✓ Saved chunk metadata to: {metadata_path}")

# Small retrieval helper for notebook use.
def retrieve_chunks(query, top_k=5):
    query_vector = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    distances, indices = vector_store.kneighbors(query_vector, n_neighbors=top_k)
    results = []
    for distance, index in zip(distances[0], indices[0]):
        result = dict(chunk_metadata[index])
        result["score"] = float(1.0 - distance)
        result["chunk_text"] = chunk_records[index]["chunk_text"]
        results.append(result)
    return results

print("\nSample retrieval check:")
sample_results = retrieve_chunks("What skills did the student develop?", top_k=3)
for result in sample_results:
    print(f"  - {result['source_file']} | {result['section_title']} | score={result['score']:.3f}")

print("\n✓ RAG embedding artifacts are ready for retrieval.")

=== EMBEDDINGS AND VECTOR STORE ===

Loaded 319 chunks from SIP_reports_section_chunks.json


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5914.75it/s]


Loaded embedding model: all-MiniLM-L6-v2


Batches: 100%|██████████| 10/10 [00:07<00:00,  1.27it/s]

Built dense embedding matrix with shape: (319, 384)
Fitted cosine nearest-neighbor vector store
✓ Saved embedding model info to: rag_assets\rag_embedding_model.json
✓ Saved embeddings to: rag_assets\rag_embeddings.npy
✓ Saved vector store to: rag_assets\rag_vector_store.joblib
✓ Saved chunk metadata to: rag_assets\rag_chunk_metadata.json

Sample retrieval check:
  - 6_SIPReport.docx | Generic Life Skills Learning Outcomes | score=0.566
  - 6_SIPReport.docx | Diploma Specific Learning Outcomes | score=0.468
  - 6_SIPReport.docx | Application of Analytics Knowledge and Skills | score=0.466

✓ RAG embedding artifacts are ready for retrieval.


In [25]:
# Cell 7 - Retrieval and generation demo using the saved RAG pipeline
# Imports the reusable script from rag_assets/ and uses it to answer a question.

import importlib
import os
import rag_assets.rag_pipeline as rag_pipeline

rag_pipeline = importlib.reload(rag_pipeline)
get_context = rag_pipeline.get_context
build_prompt = rag_pipeline.build_prompt
generate_answer = rag_pipeline.generate_answer

question = "What are the common learning outcomes for students in their SIP?"
context_k = 2
preferred_model = os.getenv("OLLAMA_MODEL")
fallback_models = tuple(
    model.strip()
    for model in os.getenv("OLLAMA_FALLBACK_MODELS", "phi3,mistral,gemma2:2b,llama3.2,llama3").split(",")
    if model.strip()
)

print("=== RETRIEVAL DEMO ===\n")
context = get_context(question, k=context_k)
print(context)

print("\n=== PROMPT PREVIEW ===\n")
prompt = build_prompt(question, context, max_words=100)
print(prompt[:1600])

print("\n=== GENERATION DEMO ===\n")
print(f"Preferred model: {preferred_model or 'default pipeline setting'}")
print(f"Fallback models: {', '.join(fallback_models) if fallback_models else 'none'}")
try:
    answer = generate_answer(
        question,
        context=context,
        context_k=context_k,
        ollama_model=preferred_model,
        fallback_models=fallback_models,
        max_context_chars=1800,
        max_answer_words=100,
        timeout_seconds=300,
    )
    print(answer)
except Exception as exc:
    print(f"Ollama generation unavailable: {exc}")
    print("You can still use the retrieved context above with any local LLM call.")

=== RETRIEVAL DEMO ===



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5522.21it/s]


[Source: 10_SIPReport.docx | Section: SIP Learning Outcomes | Score: 0.854]
SIP Learning Outcomes

---

[Source: 10_SIPReport.docx | Section: Summarise your main SIP Learning Outcomes(Generic, Diploma, Organization) | Score: 0.678]
Summarise your main SIP Learning Outcomes(Generic, Diploma, Organization)

Demonstrates the ability to communicate and work with different audiences, different walks of life and appreciate different perspectives Demonstrates the ability to analyse evidence and apply logical reasoning to develop effective and creative solutions to problems Demonstrates ability to manage own time, physical/mental health, and financial resources Demonstrates the ability to pursue own learning of new knowledge and skills Apply data exploration, transformation, and data integration techniques Apply data visualisation techniques to explore data, using appropriate methods and tools Use appropriate techniques and tools for data mining and/or predictive analysis Manage BI application

KeyboardInterrupt: 